In [9]:
from pathlib import Path

import geopandas as gpd
import h3
from shapely.geometry import Polygon
import yaml
from shapely import set_precision

with open("../config.budapest.yaml", "r") as fp:
    config = yaml.safe_load(fp)


def to_polygon(x) -> Polygon:
    return Polygon([(i[1], i[0]) for i in x])

In [4]:
area = gpd.read_file(
    f"{config['dirs']['output']}/{config['city']}/bounding_box.geojson"
)

In [16]:
Path("../data/h3/budapest_region").mkdir(exist_ok=True, parents=True)
result = {}
for res in range(6, 11):
    ids = h3.h3shape_to_cells_experimental(
        h3.geo_to_h3shape(area.geometry[0]), res, contain="overlap"
    )
    geoms = [to_polygon(h3.cell_to_boundary(i)) for i in ids]
    gdf = gpd.GeoDataFrame(
        {"h3": ids, "geometry": geoms}, geometry="geometry", crs=4326
    )
    result[res] = gdf
    gdf["geometry"] = gdf.geometry.apply(
        lambda geom: set_precision(geom, grid_size=1 / 1e7)
    )
    gdf.to_file(f"../data/h3/budapest_region/{res}.geojson")